# 🏥 Federated Learning for Healthcare Dataset
### Parts A, B & C — Implementation, Simulation & Communication Optimization
---
| | |
|---|---|
| **Algorithm** | FedAvg (McMahan et al., 2017) |
| **Framework** | Python + PyTorch |
| **Datasets** | Heart Disease (UCI), Diabetes (Pima), MNIST |
| **Part C** | Communication Optimization (Top-K, Quantization, Delta) |

---
## 📋 Table of Contents
1. [Install Dependencies](#1)
2. [Imports & Setup](#2)
3. [Part A — Neural Network Models](#3)
4. [Part B — Non-IID Dataset Loading & Split](#4)
5. [FedAvg Training Loop](#5)
6. [Run FedAvg on All Datasets](#6)
7. [Part C — Communication Optimization](#7)
8. [Results & Plots](#8)

## 1. Install Dependencies <a id='1'></a>

In [ ]:
!pip install ucimlrepo scikit-learn torch torchvision pandas matplotlib --quiet

## 2. Imports & Setup <a id='2'></a>

In [ ]:
import numpy as np
import pandas as pd
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

# Set device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {DEVICE}')

# Global config
NUM_CLIENTS  = 3
NUM_ROUNDS   = 20
LOCAL_EPOCHS = 5
LR           = 0.01
BATCH_SIZE   = 32
ALPHA        = 0.5   # Non-IID skewness (lower = more skewed)
print('✅ Config loaded')

## 3. Part A — Neural Network Models <a id='3'></a>
- **TabularNet** → Heart Disease & Diabetes (binary classification)
- **MNISTNet** → MNIST (10-class classification)

In [ ]:
class TabularNet(nn.Module):
    """3-layer FC network for tabular healthcare data."""
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),        nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.net(x)


class MNISTNet(nn.Module):
    """3-layer MLP for MNIST (784 → 256 → 128 → 10)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

print('✅ Models defined: TabularNet, MNISTNet')

## 4. Part B — Non-IID Dataset Loading & Split <a id='4'></a>
Uses **Dirichlet distribution** to create realistic non-IID splits across clients.

| Alpha (α) | Distribution |
|---|---|
| 0.1 | Highly non-IID (extreme skew) |
| 0.5 | Moderate non-IID *(default)* |
| 5.0 | Nearly IID |

In [ ]:
def non_iid_split(X, y, num_clients=3, alpha=0.5, seed=42):
    """Dirichlet non-IID split across clients."""
    np.random.seed(seed)
    client_indices = defaultdict(list)
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        np.random.shuffle(idx)
        props  = np.random.dirichlet(np.repeat(alpha, num_clients))
        splits = (props * len(idx)).astype(int)
        splits[-1] = len(idx) - splits[:-1].sum()
        start = 0
        for cid, cnt in enumerate(splits):
            client_indices[cid].extend(idx[start:start+cnt].tolist())
            start += cnt
    return [(X[client_indices[i]], y[client_indices[i]]) for i in range(num_clients)]


def print_distribution(clients_data, name):
    print(f'\n📊 {name} — Non-IID Distribution')
    print('─' * 45)
    for i, (X_c, y_c) in enumerate(clients_data):
        dist = dict(zip(*np.unique(y_c, return_counts=True)))
        print(f'  Client {i+1} | n={len(y_c):>4} | {dist}')


def plot_distribution(clients_data, name):
    fig, axes = plt.subplots(1, len(clients_data), figsize=(4*len(clients_data), 3), sharey=True)
    fig.suptitle(f'Non-IID Distribution — {name}', fontweight='bold')
    colors = cm.Set2(np.linspace(0, 1, len(np.unique(clients_data[0][1]))))
    for i, (ax, (_, y_c)) in enumerate(zip(axes, clients_data)):
        classes = sorted(np.unique(y_c))
        counts  = [np.sum(y_c == c) for c in classes]
        ax.bar([str(c) for c in classes], counts, color=colors[:len(classes)])
        ax.set_title(f'Client {i+1}\n(n={len(y_c)})')
        ax.set_xlabel('Class')
        if i == 0: ax.set_ylabel('Samples')
    plt.tight_layout()
    plt.show()

print('✅ Non-IID split utilities ready')

In [ ]:
# ── Load Heart Disease Dataset (UCI) ──────────────────────
print('❤️  Loading Heart Disease Dataset...')
try:
    from ucimlrepo import fetch_ucirepo
    hd = fetch_ucirepo(id=45)
    X_h = np.nan_to_num(hd.data.features.values.astype(np.float32))
    y_h = (hd.data.targets.values.ravel() > 0).astype(int)
except:
    print('  ⚠️ Using synthetic fallback')
    X_h = np.random.randn(303, 13).astype(np.float32)
    y_h = np.random.randint(0, 2, 303)

X_h = StandardScaler().fit_transform(X_h)
heart_clients = non_iid_split(X_h, y_h, NUM_CLIENTS, ALPHA)
print_distribution(heart_clients, 'Heart Disease')
plot_distribution(heart_clients, 'Heart Disease')

In [ ]:
# ── Load Diabetes Dataset (Pima) ──────────────────────────
print('🩸 Loading Diabetes Dataset...')
try:
    url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
    cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DPF','Age','Outcome']
    df   = pd.read_csv(url, header=None, names=cols)
    X_d  = df.drop('Outcome', axis=1).values.astype(np.float32)
    y_d  = df['Outcome'].values.astype(int)
    # Replace 0s with column mean
    X_d  = np.where(X_d == 0, np.nan, X_d)
    means = np.nanmean(X_d, axis=0)
    X_d[np.isnan(X_d)] = np.take(means, np.where(np.isnan(X_d))[1])
except:
    print('  ⚠️ Using synthetic fallback')
    X_d = np.random.randn(768, 8).astype(np.float32)
    y_d = np.random.randint(0, 2, 768)

X_d = StandardScaler().fit_transform(X_d)
diabetes_clients = non_iid_split(X_d, y_d, NUM_CLIENTS, ALPHA)
print_distribution(diabetes_clients, 'Diabetes')
plot_distribution(diabetes_clients, 'Diabetes')

In [ ]:
# ── Load MNIST Dataset ────────────────────────────────────
print('✍️  Loading MNIST Dataset...')
from torchvision import datasets, transforms
mnist_raw = datasets.MNIST(root='./data', train=True, download=True,
                            transform=transforms.ToTensor())
X_m = mnist_raw.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_m = mnist_raw.targets.numpy()
mnist_clients = non_iid_split(X_m, y_m, NUM_CLIENTS, ALPHA)
print_distribution(mnist_clients, 'MNIST')
plot_distribution(mnist_clients, 'MNIST')

## 5. FedAvg Training Loop <a id='5'></a>
Core FL functions: local training, FedAvg aggregation, evaluation.

In [ ]:
def train_local(model, X_c, y_c, local_epochs=5, lr=0.01, batch_size=32, device='cpu'):
    """Train model locally on client data."""
    model = model.to(device).train()
    loader = DataLoader(TensorDataset(
        torch.FloatTensor(X_c).to(device),
        torch.LongTensor(y_c).to(device)
    ), batch_size=batch_size, shuffle=True)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    for _ in range(local_epochs):
        for X_b, y_b in loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    return model, total_loss / (local_epochs * len(loader)), len(X_c)


def fedavg_aggregate(global_model, local_models, sample_counts):
    """Weighted average of client model weights."""
    total = sum(sample_counts)
    g_dict = {k: torch.zeros_like(v, dtype=torch.float32)
              for k, v in global_model.state_dict().items()}
    for model, n in zip(local_models, sample_counts):
        for k, v in model.state_dict().items():
            g_dict[k] += v.float() * (n / total)
    global_model.load_state_dict(g_dict)
    return global_model


def evaluate(model, X_test, y_test, device='cpu'):
    """Returns accuracy % on test set."""
    model.eval().to(device)
    with torch.no_grad():
        out   = model(torch.FloatTensor(X_test).to(device))
        preds = torch.argmax(out, dim=1)
        return (preds == torch.LongTensor(y_test).to(device)).float().mean().item() * 100


def make_test_set(clients_data, ratio=0.2, seed=42):
    """Hold out a global test set."""
    np.random.seed(seed)
    X_all = np.concatenate([x for x, _ in clients_data])
    y_all = np.concatenate([y for _, y in clients_data])
    idx   = np.random.permutation(len(y_all))
    n     = int(len(y_all) * ratio)
    return X_all[idx[:n]], y_all[idx[:n]]


def run_fedavg(clients_data, model, num_rounds=20, local_epochs=5,
               lr=0.01, batch_size=32, device='cpu', label=''):
    """Full FedAvg loop. Returns accuracy & loss history."""
    X_test, y_test = make_test_set(clients_data)
    acc_hist, loss_hist = [], []
    for rnd in range(1, num_rounds + 1):
        local_models, counts, losses = [], [], []
        for X_c, y_c in clients_data:
            lm, loss, n = train_local(copy.deepcopy(model), X_c, y_c,
                                       local_epochs, lr, batch_size, device)
            local_models.append(lm); counts.append(n); losses.append(loss)
        model = fedavg_aggregate(model, local_models, counts)
        acc   = evaluate(model, X_test, y_test, device)
        acc_hist.append(acc); loss_hist.append(np.mean(losses))
        print(f'  [{label}] Round {rnd:>2}/{num_rounds} | Loss: {np.mean(losses):.4f} | Acc: {acc:.2f}%')
    return acc_hist, loss_hist

print('✅ FedAvg functions ready')

## 6. Run FedAvg on All Datasets <a id='6'></a>

In [ ]:
# ── Heart Disease ─────────────────────────────────────────
print('❤️  FedAvg — Heart Disease')
heart_model = TabularNet(X_h.shape[1], 2)
heart_acc, heart_loss = run_fedavg(heart_clients, heart_model,
    NUM_ROUNDS, LOCAL_EPOCHS, LR, BATCH_SIZE, DEVICE, 'Heart')
print(f'\n  🏁 Final Accuracy: {heart_acc[-1]:.2f}%')

In [ ]:
# ── Diabetes ──────────────────────────────────────────────
print('🩸 FedAvg — Diabetes')
diab_model = TabularNet(X_d.shape[1], 2)
diab_acc, diab_loss = run_fedavg(diabetes_clients, diab_model,
    NUM_ROUNDS, LOCAL_EPOCHS, LR, BATCH_SIZE, DEVICE, 'Diabetes')
print(f'\n  🏁 Final Accuracy: {diab_acc[-1]:.2f}%')

In [ ]:
# ── MNIST ─────────────────────────────────────────────────
print('✍️  FedAvg — MNIST')
mnist_model = MNISTNet()
mnist_acc, mnist_loss = run_fedavg(mnist_clients, mnist_model,
    NUM_ROUNDS, LOCAL_EPOCHS, LR, BATCH_SIZE, DEVICE, 'MNIST')
print(f'\n  🏁 Final Accuracy: {mnist_acc[-1]:.2f}%')

In [ ]:
# ── Convergence Plots ─────────────────────────────────────
rounds = list(range(1, NUM_ROUNDS + 1))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('FedAvg Convergence — All Datasets', fontsize=14, fontweight='bold')

datasets_results = [
    ('Heart Disease', heart_acc, heart_loss, '#e74c3c'),
    ('Diabetes',      diab_acc,  diab_loss,  '#3498db'),
    ('MNIST',         mnist_acc, mnist_loss, '#2ecc71'),
]
for ax, (name, acc, loss, col) in zip(axes, datasets_results):
    ax2 = ax.twinx()
    ax.plot(rounds, acc,  color=col,     lw=2.5, label='Accuracy (%)')
    ax2.plot(rounds, loss, color=col, lw=2, ls='--', alpha=0.6, label='Loss')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Round'); ax.set_ylabel('Accuracy (%)', color=col)
    ax2.set_ylabel('Loss', color='gray')
    ax.set_ylim(0, 100); ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right'); ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('convergence_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot saved: convergence_plots.png')

In [ ]:
# ── Summary Table ─────────────────────────────────────────
print('\n' + '='*45)
print('   📊 ACCURACY SUMMARY')
print('='*45)
print(f"  {'Dataset':<16} {'Final Acc':>10} {'Best Acc':>10}")
print('-'*45)
for name, acc, _, _ in datasets_results:
    print(f"  {name:<16} {acc[-1]:>9.2f}% {max(acc):>9.2f}%")
print('='*45)

## 7. Part C — Communication Optimization <a id='7'></a>

| Technique | Idea | Expected Savings |
|---|---|---|
| **Top-K Sparsification** | Send only top 10% weights | ~85% |
| **8-bit Quantization** | Compress float32 → int8 | ~75% |
| **Delta Compression** | Send only weight changes | ~60–80% |

In [ ]:
# ── Communication Tracker ─────────────────────────────────
class CommTracker:
    def __init__(self): self.total = 0; self.per_round = []
    def log(self, b):   self.total += b; self.per_round.append(b)
    def kb(self):       return self.total / 1024
    def per_round_kb(self): return [b/1024 for b in self.per_round]

def model_bytes(model):
    return sum(p.numel() * 4 for p in model.parameters())

# ── Technique 1: Top-K Sparsification ────────────────────
def topk_sparsify(sd, k=0.1):
    out, sent = {}, 0
    for key, t in sd.items():
        flat = t.float().flatten()
        k_n  = max(1, int(len(flat) * k))
        _, idx = torch.topk(flat.abs(), k_n)
        sent += k_n * 8
        s = torch.zeros_like(flat); s[idx] = flat[idx]
        out[key] = s.reshape(t.shape)
    return out, sent

# ── Technique 2: Quantization ─────────────────────────────
def quantize(sd, bits=8):
    out, sent, levels = {}, 0, 2**bits - 1
    for key, t in sd.items():
        f = t.float()
        mn, mx = f.min(), f.max()
        sc = (mx - mn) / levels if (mx - mn) != 0 else 1.0
        q  = torch.round((f - mn) / sc).clamp(0, levels)
        out[key] = q * sc + mn
        sent += int(f.numel() * bits / 8) + 8
    return out, sent

# ── Technique 3: Delta Compression ───────────────────────
def delta_compress(prev, curr, thr=0.001):
    out, sent = {}, 0
    for key in curr:
        delta = curr[key].float() - prev[key].float()
        mask  = delta.abs() > thr
        sent += int(mask.sum().item()) * 8
        out[key] = prev[key].float() + delta * mask
    return out, sent

print('✅ Communication optimization utilities ready')

In [ ]:
def run_optimized(clients_data, method='baseline', num_rounds=20,
                  local_epochs=5, lr=0.01, batch_size=32, device='cpu'):
    """FedAvg with communication optimization."""
    X_test, y_test = make_test_set(clients_data)
    input_dim  = clients_data[0][0].shape[1]
    gm         = TabularNet(input_dim, 2).to(device)
    tracker    = CommTracker()
    acc_hist, loss_hist, prev_dict = [], [], None

    for rnd in range(1, num_rounds + 1):
        lms, counts, losses, rbytes = [], [], [], 0
        for X_c, y_c in clients_data:
            lm, loss, n = train_local(copy.deepcopy(gm), X_c, y_c,
                                       local_epochs, lr, batch_size, device)
            sd = lm.state_dict()
            if   method == 'topk':     comp, b = topk_sparsify(sd, 0.1)
            elif method == 'quantize': comp, b = quantize(sd, 8)
            elif method == 'delta' and prev_dict: comp, b = delta_compress(prev_dict, sd)
            else:                      comp, b = {k: v.float() for k,v in sd.items()}, model_bytes(lm)
            lm.load_state_dict(comp); lms.append(lm)
            counts.append(n); losses.append(loss); rbytes += b

        prev_dict = copy.deepcopy(gm.state_dict())
        gm = fedavg_aggregate(gm, lms, counts)
        tracker.log(rbytes)
        acc = evaluate(gm, X_test, y_test, device)
        acc_hist.append(acc); loss_hist.append(np.mean(losses))
        print(f'  [{method:>10}] Round {rnd:>2} | Acc: {acc:.2f}% | Comm: {rbytes/1024:.1f} KB')

    return acc_hist, loss_hist, tracker

print('✅ Optimized FL loop ready')

In [ ]:
# ── Run all 4 methods on Heart Disease ────────────────────
methods = ['baseline', 'topk', 'quantize', 'delta']
comm_results = {}

for m in methods:
    print(f'\n🔧 Method: {m.upper()}')
    acc, loss, tracker = run_optimized(
        heart_clients, method=m,
        num_rounds=NUM_ROUNDS, local_epochs=LOCAL_EPOCHS,
        lr=LR, batch_size=BATCH_SIZE, device=DEVICE
    )
    comm_results[m] = {'acc': acc, 'loss': loss, 'tracker': tracker}
    print(f'  ✅ Final Acc: {acc[-1]:.2f}% | Total Comm: {tracker.kb():.2f} KB')

## 8. Results & Plots <a id='8'></a>

In [ ]:
# ── 4-Panel Communication Optimization Plot ───────────────
rounds  = list(range(1, NUM_ROUNDS + 1))
colors  = {'baseline':'#e74c3c','topk':'#3498db','quantize':'#2ecc71','delta':'#f39c12'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Part C — Communication Optimization Results', fontsize=15, fontweight='bold')

# Accuracy
for m, r in comm_results.items():
    axes[0][0].plot(rounds, r['acc'], label=m.upper(), color=colors[m], lw=2.5)
axes[0][0].set_title('Accuracy vs Rounds'); axes[0][0].set_ylabel('Accuracy (%)')
axes[0][0].set_xlabel('Round'); axes[0][0].legend(); axes[0][0].grid(alpha=0.3)

# Loss
for m, r in comm_results.items():
    axes[0][1].plot(rounds, r['loss'], label=m.upper(), color=colors[m], lw=2.5)
axes[0][1].set_title('Loss vs Rounds'); axes[0][1].set_ylabel('Loss')
axes[0][1].set_xlabel('Round'); axes[0][1].legend(); axes[0][1].grid(alpha=0.3)

# Per-round comm
for m, r in comm_results.items():
    axes[1][0].plot(rounds, r['tracker'].per_round_kb(), label=m.upper(), color=colors[m], lw=2)
axes[1][0].set_title('Comm Cost per Round (KB)'); axes[1][0].set_ylabel('KB')
axes[1][0].set_xlabel('Round'); axes[1][0].legend(); axes[1][0].grid(alpha=0.3)

# Total comm bar
ms   = list(comm_results.keys())
kbs  = [comm_results[m]['tracker'].kb() for m in ms]
bars = axes[1][1].bar([m.upper() for m in ms], kbs, color=[colors[m] for m in ms], edgecolor='white')
for bar, v in zip(bars, kbs):
    axes[1][1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f'{v:.1f}', ha='center', fontweight='bold')
axes[1][1].set_title('Total Communication (KB)'); axes[1][1].set_ylabel('Total KB')
axes[1][1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comm_opt_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot saved: comm_opt_results.png')

In [ ]:
# ── Final Summary Table ───────────────────────────────────
base_kb = comm_results['baseline']['tracker'].kb()
print('\n' + '='*60)
print('   📊 COMMUNICATION OPTIMIZATION — FINAL SUMMARY')
print('='*60)
print(f"  {'Method':<12} {'Final Acc':>10} {'Total KB':>12} {'Savings':>10}")
print('-'*60)
for m, r in comm_results.items():
    acc  = r['acc'][-1]
    kb   = r['tracker'].kb()
    save = (base_kb - kb) / base_kb * 100 if m != 'baseline' else 0.0
    print(f"  {m.upper():<12} {acc:>9.2f}% {kb:>11.2f} {save:>9.1f}%")
print('='*60)
print('\n✅ All parts complete!')
print('   Saved: convergence_plots.png, comm_opt_results.png')